In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
import numpy as np
import pickle

In [ ]:
nx, ny, nz, nt = 512, 512, 100, 241
dx, dy, dz = 250.0, 250.0, 50.0 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
print('grid_vol=', grid_vol)

In [ ]:
ctl = 'goamazon_2pulse.largedom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'

In [ ]:
def total_qc(casename):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    qc_mass = np.zeros((4, nz))
    areas = ['dom', 'tl', 'tul', 'tula']
    
    for i, area in enumerate(areas):
        filename = f'{casename}/pkl/qcm_{area}.pkl'
        try:
            with open(filename, 'rb') as f:
                qcm = pickle.load(f)
            print(qcm.shape)
            # Sum over domain (no denormalization needed for qc)
            qc_mass[i,:] = qcm
        except FileNotFoundError:
            print(f'Warning: {filename} not found')
            qc_mass[i,:] = np.nan
    
    return qc_mass

In [ ]:
# Load data for both cases
qc_ctl = total_qc(ctl)
qc_ehe1 = total_qc(ehe1)

In [ ]:
# Setup height coordinate
z = np.arange(0, nz) * dz  # meters
zi = (np.arange(0, nz+1) - 0.5) * dz  # interface heights

In [ ]:
# Plot comparison of CTL vs EHE1
pop_labels = ['domain', 'tracked', 'tracked full', 'tracked full attached']
styles = ['--', ':', '-', 'None']
markers = [None, None, None, '+']

fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))

for i in range(4):
# for i in [1,2,3]:
    ax.plot(qc_ctl[i,:]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], label=f'CTL - {pop_labels[i]}', color='black')
    ax.plot(qc_ehe1[i,:]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], label=f'EHE1 - {pop_labels[i]}', color='red')
ax.set_xlabel(r'$\bar q_c$ ($10^{-3}$ g/kg)')
ax.set_ylim((0, 5))
ax.set_ylabel('Height (km)')
plt.legend(loc='upper right', fontsize=14)
plt.show()

In [ ]:
# Calculate differences (EHE1 - CTL)
qc_diff = qc_ehe1 - qc_ctl

# Create single figure
fig = plt.figure(figsize=(8, 10))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))

pop_labels = ['domain', 'tracked', 'tracked full', 'tracked full attached']
styles = ['--', ':', '-', 'None']
markers = [None, None, None, '+']

for i in range(4):
    ax.plot(qc_diff[i, :] * 1.0e3, z * 1.0e-3,
            linestyle=linestyles[i], 
            marker=markers[i],
            color='black',
            linewidth=2, label=pop_labels[i])

ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel(r'$\bar q_c$ diff ($10^{-3}$ g/kg)')
ax.set_ylabel('Height (km)')
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=12)
plt.show()